In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/xianzhezhang50/monash-train-qwen-chat/monash_train_qwen_chat.jsonl


In [2]:
!pip install -q selenium trafilatura
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -q ./google-chrome-stable_current_amd64.deb

Reading package lists...
Building dependency tree...
Reading state information...
google-chrome-stable is already the newest version (146.0.7680.80-1).
0 upgraded, 0 newly installed, 0 to remove and 134 not upgraded.


In [3]:
# ═══════════════════════════════════════════════════════════════
# Cell 3
# ═══════════════════════════════════════════════════════════════
import os, torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

hf_token = secret_value_0
print("HF_TOKEN set:", hf_token is not None)

CUDA available: True
GPU: Tesla T4
HF_TOKEN set: True


In [4]:
!pip install -q selenium

In [5]:
# ═══════════════════════════════════════════════════════════════
# Cell 4
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

# 先单独升级 pyarrow，必须在 datasets 之前装好
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyarrow>=17.0", "--upgrade"], check=True)

# 再装其余依赖
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-U", "transformers", "datasets", "accelerate",
                "peft", "trl", "bitsandbytes"], check=True)

print("Done. Please restart the kernel and run all cells again.")

Done. Please restart the kernel and run all cells again.


In [6]:
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes

In [7]:
# ═══════════════════════════════════════════════════════════════
# Cell 5 Selenium
# ═══════════════════════════════════════════════════════════════
import os
import re
import json
import time
import hashlib
import subprocess
from urllib.parse import urlparse

import trafilatura
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

def make_driver():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def fetch_html(url: str, wait_sec: int = 5) -> str:
    driver = make_driver()
    try:
        driver.get(url)
        WebDriverWait(driver, wait_sec).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(2)
        html = driver.page_source
        status = driver.execute_script("return document.readyState")
        if status != "complete":
            time.sleep(3)
            html = driver.page_source
        return html
    finally:
        driver.quit()

SEED_URLS = [
    "https://www.monash.edu/graduate-study/faqs",
    "https://www.monash.edu/graduate-research/support-and-resources/resources/faqs",
    "https://www.monash.edu/admissions/apply/help/faqs",
    "https://www.monash.edu/graduate-study/how-to-apply",
    "https://www.monash.edu/graduate-research/study/apply",
    "https://www.monash.edu/graduate-research/support-and-resources/resources/faqs/admission-scholarship",
]

BASE_DIR  = "/kaggle/working/data_monash"
RAW_DIR   = os.path.join(BASE_DIR, "raw_html")
CLEAN_DIR = os.path.join(BASE_DIR, "clean_text")
FAQ_DIR   = os.path.join(BASE_DIR, "faq_json")
os.makedirs(RAW_DIR,   exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(FAQ_DIR,   exist_ok=True)

def slugify_url(url):
    h = hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
    parsed = urlparse(url)
    path = parsed.path.strip("/").replace("/", "_")
    if not path:
        path = "home"
    path = re.sub(r"[^a-zA-Z0-9_\-]+", "_", path)
    return f"{path}_{h}"

def clean_whitespace(text):
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    return text.strip()

def extract_main_text(html, url=None):
    extracted = trafilatura.extract(
        html, url=url,
        include_links=False, include_images=False,
        include_formatting=False, favor_precision=True
    )
    if extracted and len(extracted.split()) > 80:
        return clean_whitespace(extracted)
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "noscript", "svg", "footer", "nav", "aside", "form"]):
        tag.decompose()
    main = soup.find("main") or soup.find("article") or soup.body or soup
    return clean_whitespace(main.get_text("\n", strip=True))

def extract_title(html):
    soup = BeautifulSoup(html, "lxml")
    if soup.title and soup.title.text:
        return clean_whitespace(soup.title.text)
    h1 = soup.find("h1")
    if h1:
        return clean_whitespace(h1.get_text(" ", strip=True))
    return ""

def extract_faq_pairs(html):
    soup = BeautifulSoup(html, "lxml")
    faq_pairs = []

    for d in soup.find_all("details"):
        summary = d.find("summary")
        if summary:
            q = clean_whitespace(summary.get_text(" ", strip=True))
            a = clean_whitespace(d.get_text("\n", strip=True).replace(summary.get_text(" ", strip=True), "", 1))
            if len(q) > 5 and len(a) > 20:
                faq_pairs.append({"question": q, "answer": a, "source": "details_summary"})

    for dl in soup.find_all("dl"):
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        if len(dts) == len(dds) and len(dts) > 0:
            for dt, dd in zip(dts, dds):
                q = clean_whitespace(dt.get_text(" ", strip=True))
                a = clean_whitespace(dd.get_text("\n", strip=True))
                if len(q) > 5 and len(a) > 20:
                    faq_pairs.append({"question": q, "answer": a, "source": "dl_dt_dd"})

    for tag in soup.find_all(["h2", "h3", "h4"]):
        q = clean_whitespace(tag.get_text(" ", strip=True))
        if not q or len(q) < 8:
            continue
        answer_chunks = []
        sib = tag.find_next_sibling()
        steps = 0
        while sib and steps < 8:
            if getattr(sib, "name", None) in ["h2", "h3", "h4"]:
                break
            txt = clean_whitespace(sib.get_text("\n", strip=True))
            if txt:
                answer_chunks.append(txt)
            sib = sib.find_next_sibling()
            steps += 1
        a = clean_whitespace("\n".join(answer_chunks))
        if len(a) > 20 and (
            "?" in q.lower()
            or q.lower().startswith(("how ", "what ", "can ", "do ", "does ", "is ", "are ", "when ", "where ", "who "))
        ):
            faq_pairs.append({"question": q, "answer": a, "source": "heading_blocks"})

    seen = set()
    deduped = []
    for item in faq_pairs:
        key = (item["question"], item["answer"][:200])
        if key not in seen:
            seen.add(key)
            deduped.append(item)
    return deduped

records = []
for url in tqdm(SEED_URLS):
    try:
        html = fetch_html(url)
        file_id = slugify_url(url)

        with open(os.path.join(RAW_DIR, f"{file_id}.html"), "w", encoding="utf-8") as f:
            f.write(html)

        title      = extract_title(html)
        clean_text = extract_main_text(html, url=url)
        faq_pairs  = extract_faq_pairs(html)

        with open(os.path.join(CLEAN_DIR, f"{file_id}.txt"), "w", encoding="utf-8") as f:
            f.write(clean_text)

        with open(os.path.join(FAQ_DIR, f"{file_id}.json"), "w", encoding="utf-8") as f:
            json.dump(faq_pairs, f, ensure_ascii=False, indent=2)

        records.append({
            "url": url, "title": title, "file_id": file_id,
            "clean_text_chars": len(clean_text), "faq_count": len(faq_pairs),
        })
        time.sleep(2)

    except Exception as e:
        print(f"Failed: {url} -> {e}")
        records.append({"url": url, "error": str(e)})

summary_path = os.path.join(BASE_DIR, "crawl_summary.jsonl")
with open(summary_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Done. Summary:", summary_path)

  0%|          | 0/6 [00:00<?, ?it/s]

Done. Summary: /kaggle/working/data_monash/crawl_summary.jsonl


In [8]:
# ═══════════════════════════════════════════════════════════════
# Cell 6 trafilatura + BeautifulSoup
# ═══════════════════════════════════════════════════════════════
import os, json, glob, re
import pandas as pd

BASE_DIR = "/kaggle/working/data_monash"
FAQ_DIR  = os.path.join(BASE_DIR, "faq_json")

rows, all_faqs = [], []
for fp in glob.glob(os.path.join(FAQ_DIR, "*.json")):
    with open(fp, "r", encoding="utf-8") as f:
        faq_data = json.load(f)
    file_id = os.path.basename(fp).replace(".json", "")
    rows.append({"file_id": file_id, "faq_count": len(faq_data)})
    for item in faq_data:
        all_faqs.append({
            "file_id": file_id,
            "question": item.get("question", ""),
            "answer":   item.get("answer",   ""),
            "source":   item.get("source",   "")
        })

df_faq = pd.DataFrame(all_faqs)
print("Total FAQ pairs:", len(df_faq))
display(pd.DataFrame(rows).sort_values("faq_count", ascending=False))
display(df_faq.head(20))

NAV_BAD = {
    "community", "resources", "contact", "on our site",
    "faculties", "locations", "important dates",
    "a-z index", "policies", "contact us", "campus maps",
    "jobs at monash", "staff intranet", "courses and study"
}

def normalize_text(x):
    return re.sub(r"\s+", " ", str(x).strip())

def looks_like_real_question(q):
    qn = normalize_text(q).lower()
    if len(qn) < 10 or qn in NAV_BAD:
        return False
    if "?" in qn:
        return True
    return qn.startswith(("how ", "what ", "when ", "where ", "who ", "why ",
                           "can ", "do ", "does ", "is ", "are ", "should ", "will "))

def looks_like_good_answer(a):
    an = normalize_text(a)
    if len(an) < 60:
        return False
    for p in ["contact us campus maps jobs at monash", "staff intranet", "a-z index"]:
        if p in an.lower():
            return False
    return True

df_faq["question"] = df_faq["question"].apply(normalize_text)
df_faq["answer"]   = df_faq["answer"].apply(normalize_text)
df_clean = df_faq[
    df_faq["question"].apply(looks_like_real_question) &
    df_faq["answer"].apply(looks_like_good_answer)
].copy().reset_index(drop=True)

print("Before:", len(df_faq))
print("After :", len(df_clean))
display(df_clean.head(30))

Total FAQ pairs: 85


,file_id,faq_count
2,graduate-study_faqs_dd07e68158,27
4,graduate-research_support-and-resources_resour...,21
5,admissions_apply_help_faqs_2b7377fabf,20
3,graduate-research_study_apply_513c09ca46,7
1,graduate-study_how-to-apply_87f19dbc78,6
0,graduate-research_support-and-resources_resour...,4


,file_id,question,answer,source
0,graduate-research_support-and-resources_resour...,Community,Faculties\nLocations\nDiversity and inclusion\...,details_summary
1,graduate-research_support-and-resources_resour...,Resources,Important dates\nHow to apply\nA-Z index\nPoli...,details_summary
2,graduate-research_support-and-resources_resour...,Contact,Contact us\nCampus maps\nJobs at Monash\nRecru...,details_summary
3,graduate-research_support-and-resources_resour...,On Our Site,Contact Us\nStaff intranet\nCourses and study ...,details_summary
4,graduate-study_how-to-apply_87f19dbc78,Community,Faculties\nLocations\nDiversity and inclusion\...,details_summary
5,graduate-study_how-to-apply_87f19dbc78,Resources,Important dates\nHow to apply\nA-Z index\nPoli...,details_summary
6,graduate-study_how-to-apply_87f19dbc78,Contact,Contact us\nCampus maps\nJobs at Monash\nRecru...,details_summary
7,graduate-study_how-to-apply_87f19dbc78,On Our Site,Who we are\nFind a course\nFind a researcher\n...,details_summary
8,graduate-study_how-to-apply_87f19dbc78,How to apply for a graduate course,The application process varies slightly depend...,heading_blocks
9,graduate-study_how-to-apply_87f19dbc78,What you need to provide,The online application will ask you to provide...,heading_blocks


Before: 85
After : 61


,file_id,question,answer,source
0,graduate-study_how-to-apply_87f19dbc78,How to apply for a graduate course,The application process varies slightly depend...,heading_blocks
1,graduate-study_how-to-apply_87f19dbc78,What you need to provide,The online application will ask you to provide...,heading_blocks
2,graduate-study_faqs_dd07e68158,What postgraduate studies does Monash offer?,Monash offers postgraduate studies ranging fro...,heading_blocks
3,graduate-study_faqs_dd07e68158,Does Monash offer flexible study options?,Monash offers a wide range of study options de...,heading_blocks
4,graduate-study_faqs_dd07e68158,What are the benefits of doing a master's degree?,There are many benefits of completing graduate...,heading_blocks
5,graduate-study_faqs_dd07e68158,How long does it take to complete graduate stu...,"Typically, the duration of a master’s degree b...",heading_blocks
6,graduate-study_faqs_dd07e68158,How much study commitment will I need to compl...,Full-time study load As a full-time coursework...,heading_blocks
7,graduate-study_faqs_dd07e68158,Can I reduce the length of my graduate study?,"For some courses, depending on your educationa...",heading_blocks
8,graduate-study_faqs_dd07e68158,Can I study a discipline different from my und...,Although some courses may require a specific u...,heading_blocks
9,graduate-study_faqs_dd07e68158,Can I transfer from another university?,You can apply to study at Monash after studyin...,heading_blocks


In [9]:
# ═══════════════════════════════════════════════════════════════
# Cell 7
# ═══════════════════════════════════════════════════════════════
import json

DATA_PATH = "/kaggle/input/datasets/xianzhezhang50/monash-train-qwen-chat/monash_train_qwen_chat.jsonl"

import gc
import torch

# 释放上一次运行残留的模型
try:
    del model
    del base_model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print("GPU memory freed:", torch.cuda.memory_allocated() / 1e9, "GB used")
rows = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

import random

random.seed(42)
random.shuffle(rows)

train_rows = rows          # 全部 50 条
val_rows   = []            # 不做 val split
print("train:", len(train_rows))


import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True,   # 允许 CPU offload 时保持 fp32
)

device_map = {
    "model.embed_tokens": 0,
    "model.norm": 0,
    "lm_head": 0,
    "model.layers": 0,        # 先全部放 GPU，不够再改
}

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    quantization_config=bnb_config,
    device_map="auto",        # 先让它自动分配
    offload_folder="/kaggle/working/offload",  # 磁盘 offload 目录
    offload_state_dict=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=os.environ.get("HF_TOKEN")
)



print("loaded:", model_name)

GPU memory freed: 0.0 GB used
train: 50


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

loaded: Qwen/Qwen2.5-7B-Instruct


In [10]:

# ═══════════════════════════════════════════════════════════════
# Cell 8
# ═══════════════════════════════════════════════════════════════
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 2,523,136 || all params: 7,618,139,648 || trainable%: 0.0331


In [11]:
# ═══════════════════════════════════════════════════════════════
# Cell 9 — 纯 PyTorch Dataset（仅训练集，不依赖 datasets / pyarrow）
# ═══════════════════════════════════════════════════════════════

import torch
import numpy as np
from torch.utils.data import Dataset as TorchDataset
from transformers import DataCollatorForSeq2Seq

# 1. 确保 tokenizer 有 pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id
    print("pad_token set to eos_token:", tokenizer.eos_token)

# 2. 单条样本构建函数
# 只对 assistant 回答部分计算 loss（prompt token 用 -100 遮盖）
def build_sample(row: dict, tokenizer, max_length: int = 512) -> dict:
    messages = row["messages"]

    # 完整对话文本（system + user + assistant）
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    # 仅 prompt 部分（system + user），用来定位 assistant 起始位置
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],   # 去掉最后一条 assistant 消息
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenize 完整序列
    full_enc = tokenizer(
        full_text,
        truncation=True,
        max_length=max_length,
        padding=False,
        add_special_tokens=False,
    )

    # 只 tokenize prompt，不 truncate，用于计算前缀长度
    prompt_ids = tokenizer(
        prompt_text,
        padding=False,
        add_special_tokens=False,
    )["input_ids"]

    input_ids = full_enc["input_ids"]
    attention_mask = full_enc["attention_mask"]

    # prompt 部分 label 设为 -100（不计 loss），assistant 部分保留
    prefix_len = min(len(prompt_ids), len(input_ids))
    labels = [-100] * prefix_len + input_ids[prefix_len:]
    labels = labels[:len(input_ids)]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# 3. 自定义 Dataset 类
class QADataset(TorchDataset):
    def __init__(self, rows, tokenizer, max_length=512):
        self.samples = [build_sample(r, tokenizer, max_length) for r in rows]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {k: torch.tensor(v, dtype=torch.long) for k, v in s.items()}

MAX_LEN = 512
train_dataset = QADataset(train_rows, tokenizer, MAX_LEN)

# 4. Collator：动态 padding，pad label 用 -100
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    padding=True,
    pad_to_multiple_of=8,
)

# 5. 统计
print(f"Train: {len(train_dataset)} samples")

sample = train_dataset[0]
total_tokens = len(sample["input_ids"])
answer_tokens = int((sample["labels"] != -100).sum().item())

print(
    f"Sample[0]: total={total_tokens} tokens, "
    f"prompt(masked)={total_tokens - answer_tokens}, "
    f"answer(loss)={answer_tokens}"
)

all_lens = [len(train_dataset[i]["input_ids"]) for i in range(len(train_dataset))]
print(
    f"Token length — min={min(all_lens)}, max={max(all_lens)}, "
    f"mean={np.mean(all_lens):.0f}, p95={np.percentile(all_lens, 95):.0f}"
)

Train: 50 samples
Sample[0]: total=105 tokens, prompt(masked)=65, answer(loss)=40
Token length — min=88, max=361, mean=143, p95=226


In [12]:
# ═══════════════════════════════════════════════════════════════
# Cell 10 — QLoRA 训练（用 Trainer 替代 SFTTrainer）
# ═══════════════════════════════════════════════════════════════
# SFTTrainer 内部仍会调用 datasets 的 map/format，可能触发 pyarrow 冲突。
# 直接用 transformers.Trainer + 自定义 collator，完全绕开 SFTTrainer。
# ────────────────────────────────────────────────────────────────

from transformers import TrainingArguments, Trainer

OUTPUT_DIR  = "/kaggle/working/qwen_monash_lora"
ADAPTER_DIR = f"{OUTPUT_DIR}/final_adapter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── Batch / Gradient ──────────────────────────────────────
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,       # 等效 batch_size=8
    gradient_checkpointing=True,         # 节省显存（T4 16 GB）

    # ── 学习率 ────────────────────────────────────────────────
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # ── 训练轮数 ──────────────────────────────────────────────
    num_train_epochs=5,                  # 数据集仅 45 条，多训几轮

    # ── 评估 & 保存 ───────────────────────────────────────────
    eval_strategy="no",                  # 不做 eval
    save_strategy="epoch",
    load_best_model_at_end=False, 
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ── 精度 & 日志 ───────────────────────────────────────────
    fp16=True,                           # T4 不支持 bf16
    logging_steps=5,
    report_to="none",

    # ── 其他 ─────────────────────────────────────────────────
    dataloader_pin_memory=False,
    remove_unused_columns=False,         # 自定义 Dataset 必须关闭
    optim="paged_adamw_8bit",            # QLoRA 标配优化器
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=None,      
    args=training_args,
    data_collator=collator,              # DataCollatorForSeq2Seq
)

print("Trainable parameters:")
trainer.model.print_trainable_parameters()

# ── 开始训练
trainer.train()

# ── 保存 LoRA Adapter（仅 delta 权重，不保存完整模型）
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\n✅ LoRA adapter saved → {ADAPTER_DIR}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainable parameters:
trainable params: 2,523,136 || all params: 7,618,139,648 || trainable%: 0.0331


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Step,Training Loss
5,3.530426
10,3.034986
15,2.292126
20,2.027386
25,1.944327
30,1.911674
35,1.934655



✅ LoRA adapter saved → /kaggle/working/qwen_monash_lora/final_adapter


In [13]:
# ═══════════════════════════════════════════════════════════════
# Cell 12 — 加载训练好的模型并进行多轮对话
# ═══════════════════════════════════════════════════════════════
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL  = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_DIR = "/kaggle/working/qwen_monash_lora/final_adapter"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=os.environ.get("HF_TOKEN"),
    trust_remote_code=True
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=os.environ.get("HF_TOKEN"),
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("✅ Model ready!\n")


def generate_reply(history, max_new_tokens=512, temperature=0.7, top_p=0.9):
    """根据对话历史生成回复。"""
    text = tokenizer.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 只解码新生成的 token
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# ── 多轮对话循环 ────────────────────────────────────────────────
SYSTEM_PROMPT = "You are a helpful assistant specialised in Monash University-related topics."

history = [{"role": "system", "content": SYSTEM_PROMPT}]

print("="*60)
print("  与 Monash Qwen 模型对话  |  输入 'quit' 或 'exit' 退出")
print("  输入 'reset' 清除对话历史重新开始")
print("="*60)

while True:
    try:
        user_input = input("\n你: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 对话结束。")
        break

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit", "退出"):
        print("👋 对话结束。")
        break

    if user_input.lower() in ("reset", "重置"):
        history = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("🔄 对话历史已清除，重新开始。")
        continue

    # 将用户消息追加到历史
    history.append({"role": "user", "content": user_input})

    print("模型思考中...", end="", flush=True)
    reply = generate_reply(history)
    print(f"\r助手: {reply}")

    # 将助手回复追加到历史，保持多轮上下文
    history.append({"role": "assistant", "content": reply})


Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Applying LoRA adapter...
✅ Model ready!

  与 Monash Qwen 模型对话  |  输入 'quit' 或 'exit' 退出
  输入 'reset' 清除对话历史重新开始



你:  Can I reduce the length of my graduate study?


助手: The length of time it takes to complete your course will depend on how much study you do each semester, and on how many units are required for your program.
If you have already completed some relevant undergraduate units at another institution, you may be eligible to apply for recognition of prior learning (RPL). This means that you may be able to be credited with these units and be granted exemption from some of the units required to complete your program. To find out if this is possible, please contact your school.
If you are enrolled in a coursework Master’s degree, and have been assessed as being fit and well, you can also choose to study full-time or part-time. If you study part-time, you must still complete the equivalent of 24 units per year. For further information, see Study options and arrangements.
Please note that you must be able to meet the academic requirements for your course. If you are having difficulty meeting these requirements, you should discuss them with your